# 02 - Tokenizer & Processor Modifications

**Goal**: Extend Qwen2-VL's chat template, tokenizer, and processor to support audio input.

**Maps to**: Original notebook cells 22-43

**What we do here**:
1. Create a HuggingFace repository for the modified processor
2. Define `format_data()` to convert dataset samples into OpenAI conversation format with audio
3. Modify the chat template to handle `<|audio_start|><|audio_pad|><|audio_end|>` tokens
4. Add 3 audio special tokens to the tokenizer (no embedding resize needed)
5. Push the modified processor to HuggingFace
6. Test `fetch_audio`, `process_vision_info`, and end-to-end processor pipeline

**Prerequisites**:
- Forked repos with audio modifications already committed:
  - `ZhuoyuanJiang/transformers` (branch `speech-qwen2vl`) — audio token expansion in `processing_qwen2_vl.py`
  - `ZhuoyuanJiang/Qwen3-VL` (branch `speech-qwen2vl`) — `fetch_audio` in `vision_process.py`

**Where to run**: Google Colab (free tier works for this notebook).

## 1. Environment Setup

In [ ]:
# Uncomment if running on Google Colab
# Install pinned dependencies first, then our forks LAST to prevent overwrites.
# --force-reinstall --no-deps on fork lines ensures we always get the latest code from GitHub
# (pip caches git installs and won't re-download after a runtime restart otherwise).
# Pinned to exact commit hashes for reproducibility.
!pip install -q datasets librosa matplotlib soundfile huggingface_hub
!pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -q --force-reinstall --no-deps git+https://github.com/ZhuoyuanJiang/transformers.git@e6f7d83ef88d9a686d7c019d2589ddb093440d12
!pip install -q --force-reinstall --no-deps git+https://github.com/ZhuoyuanJiang/Qwen3-VL.git@56b0756a768cc3b01cba45b01c1bc3c8cb74ea3f#subdirectory=qwen-vl-utils

In [2]:
import os
import numpy as np
import soundfile as sf
from io import BytesIO
from datasets import load_dataset
from huggingface_hub import HfApi, login
from transformers import Qwen2VLProcessor
import transformers

print(f"transformers: {transformers.__version__}")
print(f"transformers path: {transformers.__file__}")

transformers: 4.56.0.dev0
transformers path: /usr/local/lib/python3.12/dist-packages/transformers/__init__.py


In [3]:
# HuggingFace login — needed to create repo and push processor
# After logging in once, the token is cached and Run All works without prompting.
import os
from huggingface_hub import login
from huggingface_hub import HfFolder

HF_TOKEN = None

# Check for cached token (from a previous login() call)
HF_TOKEN = HfFolder.get_token()

# Try Colab Secrets
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass

# Try environment variable (server / local machine)
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to HuggingFace.")
else:
    print("No cached token found. Please paste your token below (only needed once):")
    login()

Logged in to HuggingFace.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


## 2. Create HuggingFace Repository

We create a HuggingFace repository to store the modified processor/tokenizer. This repo will later also hold the model checkpoint (in Notebook 03).

In [4]:
REPO_ID = "DanJZY/Qwen2-VL-7B-Speech"

api = HfApi()
api.create_repo(repo_id=REPO_ID, exist_ok=True)
print(f"Repository ready: https://huggingface.co/{REPO_ID}")

Repository ready: https://huggingface.co/DanJZY/Qwen2-VL-7B-Speech


## 3. Load Dataset

Same streaming approach as Notebook 01. We grab a few samples for testing the processor pipeline.

In [5]:
ds_stream = load_dataset(
    "speechbrain/LargeScaleASR",
    data_files="small/train-00000*",
    streaming=True,
    split="train"
)

# Grab 3 samples for testing
test_samples = list(ds_stream.take(3))
print(f"Loaded {len(test_samples)} test samples.")
print(f"First sample text: {test_samples[0]['text'][:80]}...")
print(f"First sample duration: {test_samples[0]['duration']:.2f}s")

Loaded 3 test samples.
First sample text: AND WHAT ABOUT INTEROPERABILITY IN THE RAIL SECTOR ARE NATIONAL BARRIERS PREVENT...
First sample duration: 17.12s


## 4. `format_data()` Function

Converts a dataset sample into the OpenAI conversation format that Qwen2-VL's chat template expects. Each sample becomes a user message (with audio + text prompt) and an assistant message (with the transcription).

In [6]:
def format_data(sample):
    """Convert a dataset sample to OpenAI conversation format with audio.
    
    Args:
        sample: A dataset sample with 'wav' (dict with 'bytes') and 'text' fields.
    
    Returns:
        dict with 'messages' key containing the conversation.
    """
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "audio", "audio": sample["wav"]["bytes"]},
                    {"type": "text", "text": "Transcribe this audio."},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": sample["text"]},
                ],
            },
        ]
    }

In [7]:
# Test format_data on the first sample
formatted = format_data(test_samples[0])

print("Formatted conversation:")
for msg in formatted["messages"]:
    print(f"  Role: {msg['role']}")
    for content in msg["content"]:
        if content["type"] == "audio":
            print(f"    [audio] {len(content['audio'])} bytes")
        else:
            text = content["text"]
            preview = text[:60] + "..." if len(text) > 60 else text
            print(f"    [text] {preview}")

Formatted conversation:
  Role: user
    [audio] 547918 bytes
    [text] Transcribe this audio.
  Role: assistant
    [text] AND WHAT ABOUT INTEROPERABILITY IN THE RAIL SECTOR ARE NATIO...


## 5. Modify Chat Template

The original Qwen2-VL chat template handles `image` and `video` content types. We add an `audio` content type that wraps audio input with special tokens:

```
<|audio_start|><|audio_pad|><|audio_end|>
```

This follows the same pattern as images (`<|vision_start|><|image_pad|><|vision_end|>`).

In [8]:
# Load the original Qwen2-VL processor
processor = Qwen2VLProcessor.from_pretrained("Qwen/Qwen2-VL-7B-Instruct")

# Print the original chat template to understand its structure
print("Original chat template:")
print(processor.chat_template)

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


Original chat template:
{% set image_count = namespace(value=0) %}{% set video_count = namespace(value=0) %}{% for message in messages %}{% if loop.first and message['role'] != 'system' %}<|im_start|>system
You are a helpful assistant.<|im_end|>
{% endif %}<|im_start|>{{ message['role'] }}
{% if message['content'] is string %}{{ message['content'] }}<|im_end|>
{% else %}{% for content in message['content'] %}{% if content['type'] == 'image' or 'image' in content or 'image_url' in content %}{% set image_count.value = image_count.value + 1 %}{% if add_vision_id %}Picture {{ image_count.value }}: {% endif %}<|vision_start|><|image_pad|><|vision_end|>{% elif content['type'] == 'video' or 'video' in content %}{% set video_count.value = video_count.value + 1 %}{% if add_vision_id %}Video {{ video_count.value }}: {% endif %}<|vision_start|><|video_pad|><|vision_end|>{% elif 'text' in content %}{{ content['text'] }}{% endif %}{% endfor %}<|im_end|>
{% endif %}{% endfor %}{% if add_generation_p

In [9]:
# Modify the chat template to add audio content type handling.
# We insert an audio branch right before the text branch in the Jinja2 template.
#
# The original template has this pattern:
#   ...video stuff...<|vision_start|><|video_pad|><|vision_end|>{% elif 'text' in content %}...
#
# We insert before the text branch:
#   {% elif content['type'] == 'audio' or 'audio' in content %}<|audio_start|><|audio_pad|><|audio_end|>

original_template = processor.chat_template

# The insertion point: right before the text content handler
text_branch = "{% elif 'text' in content %}"
audio_branch = "{% elif content['type'] == 'audio' or 'audio' in content %}<|audio_start|><|audio_pad|><|audio_end|>"

# Verify the text branch exists in the template
assert text_branch in original_template, "Could not find text branch in chat template. Template may have changed."

# Insert audio branch right before the text branch
modified_template = original_template.replace(
    text_branch,
    audio_branch + text_branch
)

processor.chat_template = modified_template
print("Modified chat template (audio branch added):")
print(processor.chat_template)

Modified chat template (audio branch added):
{% set image_count = namespace(value=0) %}{% set video_count = namespace(value=0) %}{% for message in messages %}{% if loop.first and message['role'] != 'system' %}<|im_start|>system
You are a helpful assistant.<|im_end|>
{% endif %}<|im_start|>{{ message['role'] }}
{% if message['content'] is string %}{{ message['content'] }}<|im_end|>
{% else %}{% for content in message['content'] %}{% if content['type'] == 'image' or 'image' in content or 'image_url' in content %}{% set image_count.value = image_count.value + 1 %}{% if add_vision_id %}Picture {{ image_count.value }}: {% endif %}<|vision_start|><|image_pad|><|vision_end|>{% elif content['type'] == 'video' or 'video' in content %}{% set video_count.value = video_count.value + 1 %}{% if add_vision_id %}Video {{ video_count.value }}: {% endif %}<|vision_start|><|video_pad|><|vision_end|>{% elif content['type'] == 'audio' or 'audio' in content %}<|audio_start|><|audio_pad|><|audio_end|>{% elif

## 6. Add Audio Special Tokens

We add 3 special tokens to the tokenizer:
- `<|audio_start|>` — marks the beginning of audio input
- `<|audio_pad|>` — placeholder repeated based on audio duration
- `<|audio_end|>` — marks the end of audio input

**Why no embedding resize?** The Qwen2-VL model's embedding matrix was initialized for 152,064 tokens, but only ~151,657 are defined in the tokenizer. Our 3 new tokens fill unused slots within the existing vocabulary size, so `model.resize_token_embeddings()` is not needed.

In [10]:
# Check current vocab size before adding tokens
print(f"Vocab size before: {len(processor.tokenizer)}")
print(f"Last few token IDs:")
for token in ["<|image_pad|>", "<|video_pad|>"]:
    token_id = processor.tokenizer.convert_tokens_to_ids(token)
    print(f"  {token} → ID {token_id}")

Vocab size before: 151657
Last few token IDs:
  <|image_pad|> → ID 151655
  <|video_pad|> → ID 151656


In [11]:
# Add 3 audio special tokens
audio_tokens = ["<|audio_start|>", "<|audio_pad|>", "<|audio_end|>"]
num_added = processor.tokenizer.add_special_tokens({"additional_special_tokens": audio_tokens})
print(f"Added {num_added} special tokens.")

# Verify the new token IDs
print(f"\nVocab size after: {len(processor.tokenizer)}")
print(f"Model embedding size: 152,064")
print(f"\nNew token IDs:")
for token in audio_tokens:
    token_id = processor.tokenizer.convert_tokens_to_ids(token)
    print(f"  {token} → ID {token_id}")

# Verify they are within the model's embedding size
max_token_id = max(processor.tokenizer.convert_tokens_to_ids(t) for t in audio_tokens)
assert max_token_id < 152064, f"Token ID {max_token_id} exceeds embedding size 152064!"
print(f"\nAll token IDs < 152,064 — no embedding resize needed.")

Added 3 special tokens.

Vocab size after: 151660
Model embedding size: 152,064

New token IDs:
  <|audio_start|> → ID 151657
  <|audio_pad|> → ID 151658
  <|audio_end|> → ID 151659

All token IDs < 152,064 — no embedding resize needed.


## 7. Save & Push Processor to HuggingFace

Save the modified processor (tokenizer + chat template) locally, then upload to our HuggingFace repository.

In [12]:
# Save processor locally
SAVE_DIR = "./speech_processor"
processor.save_pretrained(SAVE_DIR)
print(f"Processor saved to {SAVE_DIR}/")
print(f"Contents: {os.listdir(SAVE_DIR)}")

Processor saved to ./speech_processor/
Contents: ['special_tokens_map.json', 'video_preprocessor_config.json', 'preprocessor_config.json', 'tokenizer_config.json', 'chat_template.jinja', 'merges.txt', 'vocab.json', 'tokenizer.json', 'added_tokens.json']


In [13]:
# Push to HuggingFace
api.upload_folder(
    folder_path=SAVE_DIR,
    repo_id=REPO_ID,
)
print(f"Processor uploaded to https://huggingface.co/{REPO_ID}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._processor/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Processor uploaded to https://huggingface.co/DanJZY/Qwen2-VL-7B-Speech


## 8. Load & Test Processor from HuggingFace

Verify that the processor loads correctly from our HuggingFace repository and produces the expected audio tokens.

In [14]:
# Load processor from HuggingFace (fresh load, not from local cache)
processor_hf = Qwen2VLProcessor.from_pretrained(REPO_ID)

# Verify audio tokens exist in the loaded tokenizer
for token in ["<|audio_start|>", "<|audio_pad|>", "<|audio_end|>"]:
    token_id = processor_hf.tokenizer.convert_tokens_to_ids(token)
    print(f"{token} → ID {token_id}")
    assert token_id != processor_hf.tokenizer.unk_token_id, f"{token} mapped to UNK!"

<|audio_start|> → ID 151657
<|audio_pad|> → ID 151658
<|audio_end|> → ID 151659


In [15]:
# Apply chat template to a formatted conversation
formatted = format_data(test_samples[0])
text_output = processor_hf.apply_chat_template(
    formatted["messages"],
    tokenize=False,
    add_generation_prompt=False
)
print("Chat template output:")
print(text_output)
print()

# Tokenize and check for audio tokens in input_ids
token_ids = processor_hf.tokenizer(text_output, return_tensors="np")["input_ids"][0]
audio_pad_id = processor_hf.tokenizer.convert_tokens_to_ids("<|audio_pad|>")
audio_start_id = processor_hf.tokenizer.convert_tokens_to_ids("<|audio_start|>")
audio_end_id = processor_hf.tokenizer.convert_tokens_to_ids("<|audio_end|>")

print(f"Total tokens: {len(token_ids)}")
print(f"audio_start count: {(token_ids == audio_start_id).sum()}")
print(f"audio_pad count: {(token_ids == audio_pad_id).sum()}")
print(f"audio_end count: {(token_ids == audio_end_id).sum()}")

# Decode back to text to verify
decoded = processor_hf.tokenizer.decode(token_ids)
print(f"\nDecoded text (first 200 chars):")
print(decoded[:200])

Chat template output:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|audio_start|><|audio_pad|><|audio_end|>Transcribe this audio.<|im_end|>
<|im_start|>assistant
AND WHAT ABOUT INTEROPERABILITY IN THE RAIL SECTOR ARE NATIONAL BARRIERS PREVENTING PROGRESS IN THIS AREA AS WELL OR IS THERE AN UNWILLINGNESS ON THE PART OF THE RAIL INDUSTRY TO EMBRACE THE CONCEPT OF INTEROPERABILITY<|im_end|>


Total tokens: 89
audio_start count: 1
audio_pad count: 1
audio_end count: 1

Decoded text (first 200 chars):
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|audio_start|><|audio_pad|><|audio_end|>Transcribe this audio.<|im_end|>
<|im_start|>assistant
AND WHAT ABOUT INTEROPERABILI


## 9. Test `fetch_audio` & `process_vision_info`

Test the `fetch_audio` function and `process_vision_info` from our modified `qwen-vl-utils` fork. These handle loading audio from the conversation format and returning it as numpy arrays.

In [16]:
from qwen_vl_utils import process_vision_info
from qwen_vl_utils.vision_process import fetch_audio, AUDIO_SAMPLE_RATE

print(f"AUDIO_SAMPLE_RATE: {AUDIO_SAMPLE_RATE}")

AUDIO_SAMPLE_RATE: 16000


In [17]:
# Test fetch_audio directly with raw bytes from dataset
audio_bytes = test_samples[0]["wav"]["bytes"]
audio_array, sr = fetch_audio({"audio": audio_bytes})

print(f"fetch_audio output:")
print(f"  audio_array shape: {audio_array.shape}")
print(f"  audio_array dtype: {audio_array.dtype}")
print(f"  sample_rate: {sr}")
print(f"  duration: {len(audio_array) / sr:.2f}s")
print(f"  min/max: {audio_array.min():.4f} / {audio_array.max():.4f}")

fetch_audio output:
  audio_array shape: (273920,)
  audio_array dtype: float32
  sample_rate: 16000
  duration: 17.12s
  min/max: -0.9160 / 1.0000


In [18]:
# Test process_vision_info with a formatted conversation containing audio
formatted = format_data(test_samples[0])
image_inputs, video_inputs, audio_inputs = process_vision_info(formatted["messages"])

print(f"image_inputs: {image_inputs}")
print(f"video_inputs: {video_inputs}")
print(f"audio_inputs: {type(audio_inputs)}, length: {len(audio_inputs) if audio_inputs else 0}")

if audio_inputs:
    for i, (arr, rate) in enumerate(audio_inputs):
        print(f"  Audio {i}: shape={arr.shape}, sr={rate}, duration={len(arr)/rate:.2f}s, dtype={arr.dtype}")

image_inputs: None
video_inputs: None
audio_inputs: <class 'list'>, length: 1
  Audio 0: shape=(273920,), sr=16000, duration=17.12s, dtype=float32


## 10. End-to-End Processor Test

Full pipeline: `format_data` → `process_vision_info` → `processor.__call__`

This tests the complete flow from raw dataset sample to model-ready tensors, including:
- Audio token expansion (single `<|audio_pad|>` → repeated based on duration)
- WhisperFeatureExtractor mel spectrogram generation
- Final `BatchFeature` with `input_ids`, `attention_mask`, `audio_features`, `audio_lengths`

In [19]:
# Full pipeline test
sample = test_samples[0]

# Step 1: Format data
formatted = format_data(sample)

# Step 2: Extract audio inputs via process_vision_info
image_inputs, video_inputs, audio_inputs = process_vision_info(formatted["messages"])

# Step 3: Apply chat template to get text with audio tokens
text = processor_hf.apply_chat_template(
    formatted["messages"],
    tokenize=False,
    add_generation_prompt=False
)
print(f"Text after chat template (first 150 chars):")
print(text[:150])
print()

# Step 4: Call processor with text and audio inputs
# This triggers audio token expansion and WhisperFeatureExtractor processing
batch = processor_hf(
    text=text,
    audios=audio_inputs,
    return_tensors="np",
    padding=True,
)

print("BatchFeature contents:")
for key, value in batch.items():
    if hasattr(value, 'shape'):
        print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
    elif isinstance(value, list):
        print(f"  {key}: {value}")
    else:
        print(f"  {key}: {type(value)}")

Text after chat template (first 150 chars):
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|audio_start|><|audio_pad|><|audio_end|>Transcribe this audio.<|im_end|>
<

BatchFeature contents:
  input_ids: shape=(1, 944), dtype=int64
  attention_mask: shape=(1, 944), dtype=int64
  audio_features: shape=(1, 128, 3000), dtype=float32
  audio_lengths: shape=(1,), dtype=int64


In [20]:
# Verify audio token expansion happened correctly
audio_pad_id = processor_hf.tokenizer.convert_tokens_to_ids("<|audio_pad|>")
input_ids = batch["input_ids"][0]
num_audio_pad_tokens = (input_ids == audio_pad_id).sum()

# Expected: based on audio duration
import math
audio_array, sr = audio_inputs[0]
duration = len(audio_array) / sr
expected_tokens = min(math.ceil(duration * 50), 1500)

print(f"Audio duration: {duration:.2f}s")
print(f"Expected audio tokens: {expected_tokens}")
print(f"Actual audio_pad tokens in input_ids: {num_audio_pad_tokens}")
assert num_audio_pad_tokens == expected_tokens, f"Token count mismatch! Expected {expected_tokens}, got {num_audio_pad_tokens}"
print("Token expansion verified.")

# Verify audio_features shape
print(f"\naudio_features shape: {batch['audio_features'].shape}")
print(f"  → (num_audios={batch['audio_features'].shape[0]}, mel_bins={batch['audio_features'].shape[1]}, time_frames={batch['audio_features'].shape[2]})")
print(f"audio_lengths: {batch['audio_lengths']}")

Audio duration: 17.12s
Expected audio tokens: 856
Actual audio_pad tokens in input_ids: 856
Token expansion verified.

audio_features shape: (1, 128, 3000)
  → (num_audios=1, mel_bins=128, time_frames=3000)
audio_lengths: [856]


## 11. Cleanup

In [21]:
import torch

del processor, processor_hf, batch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Memory cleaned.")

Memory cleaned.


## Summary

**What we built**:
1. `format_data()` — converts dataset samples to OpenAI conversation format with audio
2. Modified chat template — adds `<|audio_start|><|audio_pad|><|audio_end|>` for audio content
3. Added 3 audio special tokens (IDs 151657-151659) — within existing vocab, no resize needed
4. Pushed modified processor to HuggingFace (`DanJZY/Qwen2-VL-7B-Speech`)
5. Verified end-to-end: `format_data` → `process_vision_info` → `processor.__call__` → `BatchFeature`

**What the processor now outputs** (via `BatchFeature`):
- `input_ids` — token IDs with `<|audio_pad|>` expanded based on audio duration
- `attention_mask` — standard attention mask
- `audio_features` — mel spectrograms from WhisperFeatureExtractor, shape `(num_audios, 128, 3000)` (128 mel bins for whisper-large-v3-turbo)
- `audio_lengths` — token count per audio, used by the model to split the concatenated features

**Next**: Notebook 03 — Model Architecture (add Whisper encoder + audio projector to Qwen2-VL)